# JSON mode
Same prompt as *just ask*, but `response_mime_type="application/json"` guarantees syntactically valid JSON.

In [1]:
import json
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import JSON, Markdown, display

load_dotenv()  # loads GEMINI_API_KEY from .env
client = genai.Client()

from demo_helpers import VALID_APPLICATIONS

In [2]:
transcript = "Hey Google, navigate to Ara."

## Input

In [3]:
schema_string = """
{
    "target_application": "[application]",
    "target_action": "[action]",
    "additional_parameters": {
        "[parameter_name_1]": "[parameter_value_1]",
        ...
    }
}
"""

message = f"""For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.\n
Do not include any additional information or preamble. \n
The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):\n
{json.dumps(VALID_APPLICATIONS, indent=2)}\n
Return the response as json following the schema:\n{schema_string}

transcript: {transcript}"""
print(message)

For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.

Do not include any additional information or preamble. 

The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):

{
  "maps": {
    "description": "Navigation and location search",
    "actions": {
      "navigate": {
        "destination": "str",
        "mode": "driving | walking | cycling | transit"
      },
      "search_nearby": {
        "query": "str",
        "radius_km": "float"
      },
      "show_traffic": {}
    }
  },
  "music": {
    "description": "Music streaming and playback",
    "actions": {
      "play": {
        "song": "str",
        "artist": "str",
        "playlist": "str"
      },
      "pause": {},
      "skip": {},
      "set_volume": {
        "level": "int (0-100)"
      }
    }
  },
  "messages": {
    "descript

In [4]:
config = types.GenerateContentConfig(response_mime_type="application/json")
config

GenerateContentConfig(
  response_mime_type='application/json'
)

## Model call

In [5]:
start = time.perf_counter()
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=message,
    config=config,
)
print(f"Client call took {time.perf_counter() - start:.2f}s")

Client call took 1.07s


## Output

In [8]:
print(response.text)

{
    "target_application": "maps",
    "target_action": "navigate",
    "additional_parameters": {
        "destination": "Ara"
    }
}


In [12]:
JSON(json.loads(response.text))

<IPython.core.display.JSON object>